# **Distinguiendo Parámetros vs. Hiperparámetros**

***Parámetros*** (Lo que el modelo aprende):
*   Son valores internos del modelo que se ajustan automáticamente durante el
proceso de entrenamiento (.fit()).
*   No los decides tú, los decide el algoritmo de optimización (ej. Gradiente Descendiente).Son el resultado del entrenamiento.
*   *Ejemplos*:
*   Regresión Logística: Los coeficientes ($\beta$) para cada feature.
*   Red Neuronal: Los pesos (weights) y sesgos (biases) de cada neurona.
*   Árbol de Decisión: Los puntos de corte óptimos y las features elegidas en cada nodo.
---

***Hiperparámetros*** (Lo que el modelo usa para aprender):
*   Son valores externos al modelo que tú, el científico de datos, debes definir antes de iniciar el entrenamiento.Controlan cómo debe ser el proceso de aprendizaje.
*   No se aprenden de los datos; se configuran.
*   La Hiperparametrización (HPO) es el proceso de encontrar la mejor combinación de estos valores.
---
Vamos a usar un dataset dummy para ilustrar.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import xgboost as xgb
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import Adam

# 1. Crear un Dataset Dummy
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10,
                           n_redundant=5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Datos de entrenamiento: {X_train.shape}")
print(f"Datos de prueba: {X_test.shape}")

Datos de entrenamiento: (800, 20)
Datos de prueba: (200, 20)


# **XGBoost**

In [12]:
# --- HIPERPARÁMETROS (Definidos por NOSOTROS antes del .fit()) ---
config_xgb = {
    'objective': 'binary:logistic', # Tarea a realizar
    'eval_metric': 'logloss',       # Métrica de evaluación
    'n_estimators': 500,            # Número de árboles a construir
    'max_depth': 10,                 # Profundidad máxima de cada árbol
    'learning_rate': 0.01,           # Tasa de aprendizaje (cuánto encoge cada árbol)
    'subsample': 0.8,               # Fracción de datos para entrenar cada árbol
    'colsample_bytree': 0.8         # Fracción de features para entrenar cada árbol
}

print("--- XGBoost ---")
# Instanciamos el modelo con nuestros HIPERPARÁMETROS
model_xgb = xgb.XGBClassifier(**config_xgb)

# 2. Entrenamiento: Aquí es donde el modelo APRENDE sus PARÁMETROS
# Los parámetros (las divisiones del árbol, los valores hoja) se
# están calculando internamente.
model_xgb.fit(X_train, y_train)

# Los parámetros aprendidos VIVEN dentro de 'model_xgb' (ej. model_xgb.get_booster())
# pero no son tan explícitos como en una Red Neuronal.
# Lo que SÍ podemos ver es el resultado de esos parámetros:
preds = model_xgb.predict(X_test)
print(f"Accuracy (XGBoost): {accuracy_score(y_test, preds):.4f}")

--- XGBoost ---
Accuracy (XGBoost): 0.8950


# **Neural Network**

In [13]:
# --- HIPERPARÁMETROS (Definidos por NOSOTROS antes del .fit()) ---
# La arquitectura de la red ES un conjunto de hiperparámetros.
n_capa_1 = 64              # Hiperparámetro: neuronas en la capa 1
activ_capa_1 = 'relu'      # Hiperparámetro: función de activación
n_capa_2 = 32              # Hiperparámetro: neuronas en la capa 2
activ_capa_2 = 'relu'      # Hiperparámetro: función de activación

# Hiperparámetros del proceso de entrenamiento
hp_learning_rate = 0.001
hp_batch_size = 32
hp_epochs = 10

print("\n--- Red Neuronal (Keras) ---")
model_nn = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(n_capa_1, activation=activ_capa_1), # Definido por HP
    Dense(n_capa_2, activation=activ_capa_2), # Definido por HP
    Dense(1, activation='sigmoid')            # Capa de salida
])

# Instanciamos el optimizador con su HIPERPARÁMETRO (learning_rate)
optimizer = Adam(learning_rate=hp_learning_rate)

model_nn.compile(optimizer=optimizer,
                 loss='binary_crossentropy',
                 metrics=['accuracy'])

# --- PARÁMETROS (Lo que el modelo aprenderá) ---
# Keras nos dice exactamente cuántos parámetros (pesos y sesgos)
# va a aprender el modelo.
print("Resumen del modelo: ")
model_nn.summary()

# 2. Entrenamiento: El modelo APRENDE sus PARÁMETROS (pesos y sesgos)
model_nn.fit(X_train, y_train,
             epochs=hp_epochs,
             batch_size=hp_batch_size,
             validation_data=(X_test, y_test),
             verbose=0) # verbose=0 para no llenar la salida

# Los parámetros (pesos) ahora tienen valores específicos:
# (Esto solo es para demostrar, no es necesario hacerlo siempre)
# print("Primeros pesos de la capa 1 (Parámetros):")
# print(model_nn.layers[0].get_weights()[0][:2])

loss, acc = model_nn.evaluate(X_test, y_test, verbose=0)
print(f"Accuracy (Red Neuronal): {acc:.4f}")


--- Red Neuronal (Keras) ---
Resumen del modelo: 


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,344 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,457 (13.50 KB)

 Trainable params: 3,457 (13.50 KB)

 Non-trainable params: 0 (0.00 B)

Accuracy (Red Neuronal): 0.9500


# **Impacto en Performance y Estabilidad**

Usemos el XGBoost y veamos cómo dos configuraciones distintas afectan el overfitting.

*   Modelo A (Simple / Potencial Underfit): Poca profundidad, pocos árboles.
*   Modelo B (Complejo / Potencial Overfit): Mucha profundidad, muchos árboles.

In [14]:
print("\n--- Impacto de HPs en aprendizaje ---")

# --- Modelo A: Simple (Riesgo de High-Bias / Underfitting) ---
# HPs: Poca profundidad, pocos estimadores
model_A = xgb.XGBClassifier(
    n_estimators=5,    # Pocos árboles
    max_depth=3,        # Árboles poco profundos
    learning_rate=0.01,
    random_state=42)

model_A.fit(X_train, y_train)
acc_A_train = accuracy_score(y_train, model_A.predict(X_train))
acc_A_test = accuracy_score(y_test, model_A.predict(X_test))

print(f"Modelo A (Simple):")
print(f"  Acc train: {acc_A_train:.4f}")
print(f"  Acc test:  {acc_A_test:.4f}")
print(f"  Ratio de decaímiento: {acc_A_train / acc_A_test:.4f}")


# --- Modelo B: Complejo (Riesgo de High-Variance / Overfitting) ---
# HPs: Mucha profundidad, muchos estimadores, sin regularización
model_B = xgb.XGBClassifier(
    n_estimators=1000,   # Muchos árboles
    max_depth=20,       # Árboles muy profundos
    learning_rate=0.01,
    random_state=42)

model_B.fit(X_train, y_train)
acc_B_train = accuracy_score(y_train, model_B.predict(X_train))
acc_B_test = accuracy_score(y_test, model_B.predict(X_test))

print(f"\nModelo B (Complejo):")
print(f"  Acc train: {acc_B_train:.4f}") # Probablemente sea 1.0000
print(f"  Acc test:  {acc_B_test:.4f}")  # Probablemente peor que el Modelo A
print(f"  DRatio de decaímiento: {acc_B_train / acc_B_test:.4f}")


--- Impacto de HPs en aprendizaje ---
Modelo A (Simple):
  Acc train: 0.7987
  Acc test:  0.7350
  Ratio de decaímiento: 1.0867

Modelo B (Complejo):
  Acc train: 1.0000
  Acc test:  0.9000
  DRatio de decaímiento: 1.1111


# **Algoritmos de búsqueda**

Objetivo: Encontrar la mejor combinación de HPs para nuestro XGBoost.

Setup: Primero, definimos el "espacio de búsqueda" (qué HPs queremos probar y en qué rangos).

In [15]:
!pip install scikit-optimize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.8/107.8 kB 4.4 MB/s eta 0:00:00


In [16]:
# Setup para los algoritmos de búsqueda
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, KFold
from skopt import BayesSearchCV
import time

# Usaremos un subconjunto de datos para que las búsquedas sean rápidas en clase
X_sample, _, y_sample, _ = train_test_split(X, y, train_size=0.5, random_state=42)
print(f"\nUsando {len(y_sample)} muestras para HPO...")

# Modelo base que vamos a optimizar
base_model = xgb.XGBClassifier(objective='binary:logistic', eval_metric='logloss',
                               use_label_encoder=False, random_state=42)

# Cross-validation strategy
# Usamos la misma para todos, para que sea una comparación justa
cv_strategy = KFold(n_splits=3, shuffle=True, random_state=42)


Usando 500 muestras para HPO...


# ***Grid Search***

Cómo funciona: Prueba todas las combinaciones posibles que le damos.

Pros: Exhaustivo. Si la solución óptima está en la rejilla, la encontrará.

Contras: Extremadamente lento. Crece exponencialmente con más HPs o más valores.

In [22]:
# Espacio de búsqueda para Grid Search (pequeño a propósito)
param_grid = {
    'n_estimators': [50, 500],         # 2 valores
    'max_depth': [2, 10],                # 2 valores
    'learning_rate': [0.01, 0.2]        # 2 valores
}
# Total de modelos a entrenar: 2 * 2 * 2 = 8 combinaciones
# Multiplicado por los folds de CV (3): 8 * 3 = 24 entrenamientos

print("\n--- 1. Ejecutando Grid Search ---")
start_time = time.time()
grid_search = GridSearchCV(
    estimator=base_model,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1, # Usar todos los cores
    verbose=1
)

grid_search.fit(X_sample, y_sample)
end_time = time.time()

print(f"Grid Search tomó {end_time - start_time:.2f} segundos")
print(f"Mejor Puntuación (Acc): {grid_search.best_score_:.4f}")
print(f"Mejores Hiperparámetros: {grid_search.best_params_}")


--- 1. Ejecutando Grid Search ---
Fitting 3 folds for each of 8 candidates, totalling 24 fits
Grid Search tomó 6.81 segundos
Mejor Puntuación (Acc): 0.9260
Mejores Hiperparámetros: {'learning_rate': 0.2, 'max_depth': 10, 'n_estimators': 500}


# ***Random Search***
Cómo funciona: En lugar de probar todo, toma muestras aleatorias del espacio de búsqueda durante un número fijo de iteraciones.

Pros: Mucho más rápido. Sorprendentemente efectivo (a menudo los HPs más importantes se encuentran rápido).

Contras: No garantiza encontrar el óptimo, solo uno "suficientemente bueno".

In [18]:
from scipy.stats import randint, uniform

# Espacio de búsqueda para Random Search (rangos continuos o discretos)
# ¡Podemos definir espacios mucho más grandes!
param_dist = {
    'n_estimators': randint(50, 500), # Entero aleatorio entre 50 y 500
    'max_depth': randint(2, 10),      # Entero aleatorio entre 2 y 10
    'learning_rate': uniform(0.01, 0.2), # Flotante aleatorio entre 0.01 y 0.21
    'subsample': uniform(0.7, 0.3)      # (0.7 a 1.0)
}

# n_iter: cuántas combinaciones aleatorias probar
# 10 iteraciones * 3 folds = 30 entrenamientos
# (Mucho menos que un GridSearch sobre este espacio)

print("\n--- 2. Ejecutando Random Search ---")
start_time = time.time()
random_search = RandomizedSearchCV(
    estimator=base_model,
    param_distributions=param_dist,
    n_iter=10, # Probar 10 combinaciones aleatorias
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

random_search.fit(X_sample, y_sample)
end_time = time.time()

print(f"Random Search tomó {end_time - start_time:.2f} segundos")
print(f"Mejor Puntuación (Acc): {random_search.best_score_:.4f}")
print(f"Mejores Hiperparámetros: {random_search.best_params_}")


--- 2. Ejecutando Random Search ---
Fitting 3 folds for each of 10 candidates, totalling 30 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:14:17] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Random Search tomó 10.57 segundos
Mejor Puntuación (Acc): 0.9180
Mejores Hiperparámetros: {'learning_rate': np.float64(0.1284829137724085), 'max_depth': 4, 'n_estimators': 356, 'subsample': np.float64(0.9040922615763338)}


# ***Bayesian Optimization***
Cómo funciona: Es una búsqueda inteligente. Utiliza los resultados de las primeras iteraciones para construir un "mapa" de la función de puntuación (un modelo sustituto). Luego, decide inteligentemente qué combinación probar a continuación para maximizar la probabilidad de mejorar.

Pros: Mucho más eficiente que Grid o Random. Converge a buenos resultados en menos iteraciones.

Contras: Más complejo de configurar. Gasta un poco de tiempo "pensando" (calculando el siguiente punto).

In [19]:
# Usaremos scikit-optimize (BayesSearchCV) que tiene una API similar a scikit-learn

# Espacio de búsqueda para Bayes (similar a Random, pero con tipos específicos)
# (pip install scikit-optimize)
from skopt.space import Real, Integer

bayes_space = {
    'n_estimators': Integer(50, 500),
    'max_depth': Integer(2, 10),
    'learning_rate': Real(0.01, 0.2, 'log-uniform'), # 'log-uniform' es mejor para tasas
    'subsample': Real(0.7, 1.0, 'uniform')
}

print("\n--- 3. Ejecutando Bayesian Optimization (BayesSearchCV) ---")
start_time = time.time()
bayes_search = BayesSearchCV(
    estimator=base_model,
    search_spaces=bayes_space,
    n_iter=10, # Probar 10 iteraciones (inteligentes)
    cv=cv_strategy,
    scoring='accuracy',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

bayes_search.fit(X_sample, y_sample)
end_time = time.time()

print(f"Bayesian Search tomó {end_time - start_time:.2f} segundos")
print(f"Mejor Puntuación (Accu): {bayes_search.best_score_:.4f}")
print(f"Mejores Hiperparámetros: {bayes_search.best_params_}")


--- 3. Ejecutando Bayesian Optimization (BayesSearchCV) ---
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits
Fitting 3 folds for each of 1 candidates, totalling 3 fits


/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [01:15:12] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


Bayesian Search tomó 13.29 segundos
Mejor Puntuación (Accu): 0.9160
Mejores Hiperparámetros: OrderedDict({'learning_rate': 0.03416312199509112, 'max_depth': 8, 'n_estimators': 470, 'subsample': 0.7947398780461146})


# ***Evolutionary***

Cómo funciona: Usa conceptos de selección natural (Algoritmos Genéticos). Genera una "población" de pipelines (modelos + HPs). Los evalúa, se "reproduce" (combina) los mejores, introduce "mutaciones" (cambios aleatorios) y repite por generaciones.

Pros: Puede descubrir pipelines y HPs excelentes que un humano no habría pensado.

Contras: Puede ser muy lento. Menos control sobre el espacio de búsqueda.

In [20]:
!pip install optuna cmaes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.0/73.0 kB 5.1 MB/s eta 0:00:00


In [21]:
# Necesitarás instalar optuna: pip install optuna
import optuna
from optuna.samplers import CmaEsSampler # Importamos el sampler evolutivo
from sklearn.model_selection import cross_val_score
import time
import xgboost as xgb
import warnings

# Optuna puede ser muy verboso
optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=UserWarning) # Ignorar warnings de XGBoost

# 1. Definir la Función "Objective"
# Esta es la función que Optuna intentará maximizar
def objective(trial):

    # 2. Definir el espacio de búsqueda usando el objeto 'trial'
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'max_depth': trial.suggest_int('max_depth', 2, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0)
    }

    # 3. Instanciar el modelo con los HPs del 'trial' actual
    model = xgb.XGBClassifier(**param, random_state=42)

    # 4. Evaluar el modelo (usando la misma estrategia de CV que los demás)
    score = cross_val_score(model, X_sample, y_sample,
                            cv=cv_strategy,
                            scoring='accuracy',
                            n_jobs=-1)

    # 5. Retornar la métrica a maximizar (el accuracy promedio)
    return score.mean()

# --- 4. Ejecutando Evolutionary Strategy (Optuna + CMA-ES) ---
print("\n--- 4. Ejecutando Evolutionary Strategy (Optuna + CMA-ES) ---")
start_time = time.time()

# 6. Configurar el Sampler (El "algoritmo")
# Aquí le decimos a Optuna que use la estrategia evolutiva CMA-ES
sampler = CmaEsSampler(seed=42)

# 7. Crear el "Estudio"
# Le decimos que queremos maximizar la métrica
study = optuna.create_study(direction='maximize', sampler=sampler)

# 8. Ejecutar la optimización
# n_trials es como n_iter en RandomSearch
study.optimize(objective, n_trials=10, show_progress_bar=True)

end_time = time.time()

print(f"Optuna (CMA-ES) tomó {end_time - start_time:.2f} segundos")
print(f"Mejor Puntuación (Acc): {study.best_value:.4f}")
print(f"Mejores Hiperparámetros: {study.best_params}")


--- 4. Ejecutando Evolutionary Strategy (Optuna + CMA-ES) ---


  0%|          | 0/10 [00:00<?, ?it/s]

Optuna (CMA-ES) tomó 20.40 segundos
Mejor Puntuación (Acc): 0.9140
Mejores Hiperparámetros: {'n_estimators': 218, 'max_depth': 10, 'learning_rate': 0.08960785365368121, 'subsample': 0.8795975452591109, 'colsample_bytree': 0.7468055921327309}
